# 05 — Data Analysis: 20 Preguntas de Negocio (Snowpark DataFrames)

Todas las consultas se ejecutan contra `ANALYTICS.OBT_TRIPS` usando **Snowpark DataFrames**.
Las operaciones se ejecutan **dentro del motor de Snowflake** y solo los resultados se retornan al notebook.

### Performance (Sección 12)
- **Sin índices ni clustering** en `analytics.OBT_TRIPS`.
- Cada pregunta registra el **tiempo de ejecución** para documentar el rendimiento base.
- Snowflake utiliza únicamente su **micro-partitioning automático**.

## Inicialización

In [39]:
import os, time
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark import Window

print("=" * 70)
print("CELDA 1: Inicialización Snowpark + Carga OBT")
print("=" * 70)

connection_parameters = {
    "account":   os.environ["SF_ACCOUNT"],
    "user":      os.environ["SF_USER"],
    "password":  os.environ["SF_PASSWORD"],
    "database":  os.environ["SF_DATABASE"],
    "schema":    os.environ["SF_SCHEMA_ANALYTICS"],
    "warehouse": os.environ["SF_WAREHOUSE"],
    "role":      os.environ["SF_ROLE"],
}

session = Session.builder.configs(connection_parameters).create()

# Referencia lazy a OBT (no descarga datos, todo corre en Snowflake)
obt = session.table("OBT_TRIPS")

row_count = obt.count()
print(f"✓ Snowpark Session creada")
print(f"✓ OBT_TRIPS: {row_count:,} filas")
print(f"✓ Columnas: {len(obt.columns)}")
print("✓ Sin índices ni clustering en la tabla")
print("=" * 70)

# Diccionario para registrar tiempos
timings = {}

def timed_show(label, df, n=20):
    """Ejecuta .show() midiendo el tiempo y lo registra."""
    t0 = time.time()
    df.show(n)
    elapsed = round(time.time() - t0, 2)
    timings[label] = elapsed
    print(f"⏱  Tiempo de ejecución ({label}): {elapsed} s")
    return elapsed

CELDA 1: Inicialización Snowpark + Carga OBT
✓ Snowpark Session creada
✓ OBT_TRIPS: 815,553,850 filas
✓ Columnas: 43
✓ Sin índices ni clustering en la tabla


---
## (a) Top 10 zonas de pickup por volumen mensual

In [40]:
print("=" * 70)
print("(a) Top 10 zonas de pickup por volumen mensual")
print("=" * 70)

qa = (obt
      .filter(F.col("PU_ZONE").is_not_null())
      .group_by("PU_ZONE", "PU_BOROUGH")
      .agg(
          F.count("*").alias("TOTAL_TRIPS"),
          F.count_distinct(F.col("YEAR") * 100 + F.col("MONTH")).alias("NUM_MONTHS")
      )
      .with_column("AVG_MONTHLY_TRIPS", F.round(F.col("TOTAL_TRIPS") / F.col("NUM_MONTHS"), 0))
      .drop("NUM_MONTHS")
      .sort(F.col("TOTAL_TRIPS").desc())
      .limit(10))

timed_show("a", qa, 10)
print("✓ Pregunta (a) completada")
print("=" * 70)

(a) Top 10 zonas de pickup por volumen mensual
-------------------------------------------------------------------------------------
|"PU_ZONE"                     |"PU_BOROUGH"  |"TOTAL_TRIPS"  |"AVG_MONTHLY_TRIPS"  |
-------------------------------------------------------------------------------------
|Upper East Side South         |Manhattan     |31133959       |237664               |
|Midtown Center                |Manhattan     |29022570       |221546               |
|Upper East Side North         |Manhattan     |28307063       |214447               |
|Penn Station/Madison Sq West  |Manhattan     |25456622       |194325               |
|Midtown East                  |Manhattan     |25297839       |196107               |
|Times Sq/Theatre District     |Manhattan     |24256301       |185163               |
|Murray Hill                   |Manhattan     |23348672       |178234               |
|JFK Airport                   |Queens        |22792577       |172671               |
|Union 

## (b) Top 10 zonas de dropoff por volumen mensual

In [41]:
print("=" * 70)
print("(b) Top 10 zonas de dropoff por volumen mensual")
print("=" * 70)

qb = (obt
      .filter(F.col("DO_ZONE").is_not_null())
      .group_by("DO_ZONE", "DO_BOROUGH")
      .agg(
          F.count("*").alias("TOTAL_TRIPS"),
          F.count_distinct(F.col("YEAR") * 100 + F.col("MONTH")).alias("NUM_MONTHS")
      )
      .with_column("AVG_MONTHLY_TRIPS", F.round(F.col("TOTAL_TRIPS") / F.col("NUM_MONTHS"), 0))
      .drop("NUM_MONTHS")
      .sort(F.col("TOTAL_TRIPS").desc())
      .limit(10))

timed_show("b", qb, 10)
print("✓ Pregunta (b) completada")
print("=" * 70)

(b) Top 10 zonas de dropoff por volumen mensual
-------------------------------------------------------------------------------------
|"DO_ZONE"                     |"DO_BOROUGH"  |"TOTAL_TRIPS"  |"AVG_MONTHLY_TRIPS"  |
-------------------------------------------------------------------------------------
|Upper East Side North         |Manhattan     |29672474       |224791               |
|Midtown Center                |Manhattan     |28051010       |212508               |
|Upper East Side South         |Manhattan     |27813109       |210705               |
|Murray Hill                   |Manhattan     |23460759       |177733               |
|Times Sq/Theatre District     |Manhattan     |22814728       |172839               |
|Midtown East                  |Manhattan     |22346399       |169291               |
|Clinton East                  |Manhattan     |20159142       |152721               |
|Lincoln Square East           |Manhattan     |19883272       |150631               |
|Union

## (c) Evolución mensual de total_amount y tip_pct por borough

In [42]:
print("=" * 70)
print("(c) Evolución mensual de total_amount y tip_pct por borough")
print("=" * 70)

qc = (obt
      .filter(F.col("PU_BOROUGH").is_not_null())
      .group_by("PU_BOROUGH", "YEAR", "MONTH")
      .agg(
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TOTAL_AMOUNT"),
          F.round(F.avg("TIP_PCT"), 2).alias("AVG_TIP_PCT"),
          F.count("*").alias("TRIPS")
      )
      .sort("PU_BOROUGH", "YEAR", "MONTH"))

timed_show("c", qc, 100)
print("✓ Pregunta (c) completada")
print("=" * 70)

(c) Evolución mensual de total_amount y tip_pct por borough
----------------------------------------------------------------------------------
|"PU_BOROUGH"  |"YEAR"  |"MONTH"  |"AVG_TOTAL_AMOUNT"  |"AVG_TIP_PCT"  |"TRIPS"  |
----------------------------------------------------------------------------------
|Bronx         |2015    |1        |12.55               |4.81           |91712    |
|Bronx         |2015    |2        |12.74               |3.1            |122509   |
|Bronx         |2015    |3        |12.87               |3.1            |137872   |
|Bronx         |2015    |4        |13.4                |3.66           |119999   |
|Bronx         |2015    |5        |13.84               |4.88           |135925   |
|Bronx         |2015    |6        |13.92               |4.97           |119832   |
|Bronx         |2015    |7        |14.18               |9.4            |108313   |
|Bronx         |2015    |8        |14.41               |7.76           |100994   |
|Bronx         |2015    |9 

## (d) Ticket promedio (avg total_amount) por service_type y mes

In [43]:
print("=" * 70)
print("(d) Ticket promedio por service_type y mes")
print("=" * 70)

qd = (obt
      .group_by("SERVICE_TYPE", "YEAR", "MONTH")
      .agg(
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TICKET"),
          F.count("*").alias("TRIPS")
      )
      .sort("SERVICE_TYPE", "YEAR", "MONTH"))

timed_show("d", qd, 100)
print("✓ Pregunta (d) completada")
print("=" * 70)

(d) Ticket promedio por service_type y mes
---------------------------------------------------------------
|"SERVICE_TYPE"  |"YEAR"  |"MONTH"  |"AVG_TICKET"  |"TRIPS"   |
---------------------------------------------------------------
|green           |2015    |1        |14.78         |1508493   |
|green           |2015    |2        |14.49         |1574830   |
|green           |2015    |3        |14.58         |1722574   |
|green           |2015    |4        |14.81         |1664394   |
|green           |2015    |5        |15.25         |1786848   |
|green           |2015    |6        |14.96         |1638868   |
|green           |2015    |7        |14.91         |1541671   |
|green           |2015    |8        |14.93         |1532343   |
|green           |2015    |9        |15.03         |1494927   |
|green           |2015    |10       |14.88         |1630536   |
|green           |2015    |11       |14.7          |1529984   |
|green           |2015    |12       |14.74         |1608297  

## (e) Viajes por hora del día y día de semana (picos)

In [44]:
print("=" * 70)
print("(e) Viajes por hora del día y día de semana")
print("=" * 70)

qe = (obt
      .group_by("DAY_OF_WEEK", "PICKUP_HOUR")
      .agg(F.count("*").alias("TRIPS"))
      .sort("DAY_OF_WEEK", "PICKUP_HOUR"))

timed_show("e", qe, 200)
print("✓ Pregunta (e) completada")
print("=" * 70)

(e) Viajes por hora del día y día de semana
-------------------------------------------
|"DAY_OF_WEEK"  |"PICKUP_HOUR"  |"TRIPS"  |
-------------------------------------------
|0              |0              |6230924  |
|0              |1              |5480991  |
|0              |2              |4251405  |
|0              |3              |3252361  |
|0              |4              |2116223  |
|0              |5              |1026574  |
|0              |6              |1145506  |
|0              |7              |1573929  |
|0              |8              |2330383  |
|0              |9              |3464904  |
|0              |10             |4669024  |
|0              |11             |5377156  |
|0              |12             |5894413  |
|0              |13             |5998038  |
|0              |14             |6108308  |
|0              |15             |5976502  |
|0              |16             |5897995  |
|0              |17             |6059677  |
|0              |18             

## (f) p50 / p90 de trip_duration_min por borough de pickup

In [45]:
print("=" * 70)
print("(f) p50 / p90 de trip_duration_min por borough")
print("=" * 70)

# Snowpark: percentiles via call_builtin para APPROX_PERCENTILE
qf = (obt
      .filter(
          F.col("PU_BOROUGH").is_not_null() &
          F.col("TRIP_DURATION_MIN").is_not_null() &
          (F.col("TRIP_DURATION_MIN") >= 0)
      )
      .group_by("PU_BOROUGH")
      .agg(
          F.round(F.call_builtin("APPROX_PERCENTILE", F.col("TRIP_DURATION_MIN"), 0.50), 2).alias("P50_DURATION"),
          F.round(F.call_builtin("APPROX_PERCENTILE", F.col("TRIP_DURATION_MIN"), 0.90), 2).alias("P90_DURATION"),
          F.count("*").alias("TRIPS")
      )
      .sort(F.col("P50_DURATION").desc()))

timed_show("f", qf)
print("✓ Pregunta (f) completada")
print("=" * 70)

(f) p50 / p90 de trip_duration_min por borough
---------------------------------------------------------------
|"PU_BOROUGH"   |"P50_DURATION"  |"P90_DURATION"  |"TRIPS"    |
---------------------------------------------------------------
|Queens         |25.0            |55.0            |70517851   |
|Staten Island  |23.0            |70.7            |54276      |
|Bronx          |14.0            |39.07           |5321635    |
|Brooklyn       |13.0            |33.05           |34560554   |
|Unknown        |11.0            |28.44           |8858403    |
|Manhattan      |11.0            |25.0            |695466662  |
|NaN            |1.0             |60.0            |686720     |
|EWR            |0.0             |2.0             |73793      |
---------------------------------------------------------------

⏱  Tiempo de ejecución (f): 9.41 s
✓ Pregunta (f) completada


## (g) avg_speed_mph por franja horaria (6–9, 17–20) y borough

In [46]:
print("=" * 70)
print("(g) avg_speed_mph por franja horaria y borough")
print("=" * 70)

qg = (obt
      .filter(F.col("PU_BOROUGH").is_not_null() & F.col("AVG_SPEED_MPH").is_not_null())
      .with_column("TIME_SLOT",
          F.when(F.col("PICKUP_HOUR").between(6, 9), F.lit("Morning Rush (6-9)"))
           .when(F.col("PICKUP_HOUR").between(17, 20), F.lit("Evening Rush (17-20)"))
           .otherwise(F.lit("Other Hours")))
      .group_by("PU_BOROUGH", "TIME_SLOT")
      .agg(
          F.round(F.avg("AVG_SPEED_MPH"), 2).alias("AVG_SPEED"),
          F.count("*").alias("TRIPS")
      )
      .sort("PU_BOROUGH", "TIME_SLOT"))

timed_show("g", qg, 30)
print("✓ Pregunta (g) completada")
print("=" * 70)

(g) avg_speed_mph por franja horaria y borough
------------------------------------------------------------------
|"PU_BOROUGH"   |"TIME_SLOT"           |"AVG_SPEED"  |"TRIPS"    |
------------------------------------------------------------------
|Bronx          |Evening Rush (17-20)  |42.3         |963837     |
|Bronx          |Morning Rush (6-9)    |134.86       |1053637    |
|Bronx          |Other Hours           |86.1         |2956651    |
|Brooklyn       |Evening Rush (17-20)  |20.26        |7509311    |
|Brooklyn       |Morning Rush (6-9)    |49.31        |4407468    |
|Brooklyn       |Other Hours           |27.73        |21833244   |
|EWR            |Evening Rush (17-20)  |183.8        |2645       |
|EWR            |Morning Rush (6-9)    |163.19       |1949       |
|EWR            |Other Hours           |174.98       |7046       |
|Manhattan      |Evening Rush (17-20)  |15.56        |167752833  |
|Manhattan      |Morning Rush (6-9)    |19.07        |100147580  |
|Manhattan     

## (h) Participación por payment_type_desc y su relación con tip_pct

In [47]:
print("=" * 70)
print("(h) Participación por payment_type y relación con tip_pct")
print("=" * 70)

w_total = Window.partition_by()

qh = (obt
      .group_by("PAYMENT_TYPE_DESC")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("TIP_PCT"), 2).alias("AVG_TIP_PCT"),
          F.round(F.avg("TIP_AMOUNT"), 2).alias("AVG_TIP_AMOUNT")
      )
      .with_column("PCT_OF_TOTAL",
          F.round(100.0 * F.col("TRIPS") / F.sum("TRIPS").over(w_total), 2))
      .sort(F.col("TRIPS").desc()))

timed_show("h", qh)
print("✓ Pregunta (h) completada")
print("=" * 70)

(h) Participación por payment_type y relación con tip_pct
---------------------------------------------------------------------------------------
|"PAYMENT_TYPE_DESC"  |"TRIPS"    |"AVG_TIP_PCT"  |"AVG_TIP_AMOUNT"  |"PCT_OF_TOTAL"  |
---------------------------------------------------------------------------------------
|Credit card          |547213355  |25.44          |3.07              |67.10           |
|Cash                 |237533691  |0.0            |0.0               |29.13           |
|Other                |22906755   |5.08           |8.7               |2.81            |
|No charge            |4076388    |0.06           |0.04              |0.50            |
|Dispute              |3820692    |0.07           |0.03              |0.47            |
|Unknown              |2969       |1.96           |0.36              |0.00            |
---------------------------------------------------------------------------------------

⏱  Tiempo de ejecución (h): 0.25 s
✓ Pregunta (h) completada


## (i) ¿Qué rate_code_desc concentran mayor trip_distance y total_amount?

In [48]:
print("=" * 70)
print("(i) rate_code_desc: mayor trip_distance y total_amount")
print("=" * 70)

qi = (obt
      .filter(F.col("RATE_CODE_DESC").is_not_null())
      .group_by("RATE_CODE_DESC")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("TRIP_DISTANCE"), 2).alias("AVG_DISTANCE"),
          F.round(F.sum("TRIP_DISTANCE"), 0).alias("TOTAL_DISTANCE"),
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TOTAL_AMOUNT"),
          F.round(F.sum("TOTAL_AMOUNT"), 0).alias("SUM_TOTAL_AMOUNT")
      )
      .sort(F.col("SUM_TOTAL_AMOUNT").desc()))

timed_show("i", qi)
print("✓ Pregunta (i) completada")
print("=" * 70)

(i) rate_code_desc: mayor trip_distance y total_amount
----------------------------------------------------------------------------------------------------------------
|"RATE_CODE_DESC"    |"TRIPS"    |"AVG_DISTANCE"  |"TOTAL_DISTANCE"  |"AVG_TOTAL_AMOUNT"  |"SUM_TOTAL_AMOUNT"  |
----------------------------------------------------------------------------------------------------------------
|Standard rate       |764241903  |3.5             |2676408638.0      |16.77               |12815157048.0       |
|JFK                 |19139002   |19.08           |365104702.0       |70.89               |1356816808.0        |
|Unknown             |24565225   |31.96           |785061283.0       |26.77               |657707201.0         |
|Negotiated fare     |5145095    |6.11            |31454463.0        |56.22               |289257589.0         |
|Newark              |1724185    |15.85           |27323158.0        |91.38               |157547422.0         |
|Nassau/Westchester  |731657     |27.11  

## (j) Mix yellow vs green por mes y borough

In [49]:
print("=" * 70)
print("(j) Mix yellow vs green por mes y borough")
print("=" * 70)

w_part = Window.partition_by("PU_BOROUGH", "YEAR", "MONTH")

qj = (obt
      .filter(F.col("PU_BOROUGH").is_not_null())
      .group_by("PU_BOROUGH", "YEAR", "MONTH", "SERVICE_TYPE")
      .agg(F.count("*").alias("TRIPS"))
      .with_column("PCT_SHARE",
          F.round(100.0 * F.col("TRIPS") / F.sum("TRIPS").over(w_part), 2))
      .sort("PU_BOROUGH", "YEAR", "MONTH", "SERVICE_TYPE"))

timed_show("j", qj, 100)
print("✓ Pregunta (j) completada")
print("=" * 70)

(j) Mix yellow vs green por mes y borough
----------------------------------------------------------------------------
|"PU_BOROUGH"  |"YEAR"  |"MONTH"  |"SERVICE_TYPE"  |"TRIPS"  |"PCT_SHARE"  |
----------------------------------------------------------------------------
|Bronx         |2015    |1        |green           |91712    |100.00       |
|Bronx         |2015    |2        |green           |122509   |100.00       |
|Bronx         |2015    |3        |green           |137872   |100.00       |
|Bronx         |2015    |4        |green           |119999   |100.00       |
|Bronx         |2015    |5        |green           |125191   |92.10        |
|Bronx         |2015    |5        |yellow          |10734    |7.90         |
|Bronx         |2015    |6        |green           |110526   |92.23        |
|Bronx         |2015    |6        |yellow          |9306     |7.77         |
|Bronx         |2015    |7        |green           |99409    |91.78        |
|Bronx         |2015    |7        

## (k) Top 20 flujos PU→DO por volumen y su ticket promedio

In [50]:
print("=" * 70)
print("(k) Top 20 flujos PU→DO por volumen y ticket promedio")
print("=" * 70)

qk = (obt
      .filter(F.col("PU_ZONE").is_not_null() & F.col("DO_ZONE").is_not_null())
      .group_by("PU_ZONE", "DO_ZONE")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TICKET"),
          F.round(F.avg("TRIP_DISTANCE"), 2).alias("AVG_DISTANCE")
      )
      .sort(F.col("TRIPS").desc())
      .limit(20))

timed_show("k", qk, 20)
print("✓ Pregunta (k) completada")
print("=" * 70)

(k) Top 20 flujos PU→DO por volumen y ticket promedio
---------------------------------------------------------------------------------------------------------
|"PU_ZONE"                     |"DO_ZONE"                     |"TRIPS"  |"AVG_TICKET"  |"AVG_DISTANCE"  |
---------------------------------------------------------------------------------------------------------
|NaN                           |NaN                           |6859819  |17.91         |7.47            |
|Upper East Side South         |Upper East Side North         |4361744  |10.57         |1.11            |
|Upper East Side North         |Upper East Side South         |3722262  |11.46         |1.15            |
|Upper East Side North         |Upper East Side North         |3426614  |8.73          |0.63            |
|Upper East Side South         |Upper East Side South         |3303489  |9.3           |0.67            |
|Upper West Side South         |Upper West Side North         |1919235  |9.08          |0.89      

## (l) Distribución de passenger_count y efecto en total_amount

In [51]:
print("=" * 70)
print("(l) Distribución de passenger_count y efecto en total_amount")
print("=" * 70)

w_all = Window.partition_by()

ql = (obt
      .filter(F.col("PASSENGER_COUNT").is_not_null())
      .group_by("PASSENGER_COUNT")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TOTAL_AMOUNT"),
          F.round(F.avg("TRIP_DISTANCE"), 2).alias("AVG_DISTANCE")
      )
      .with_column("PCT_OF_TOTAL",
          F.round(100.0 * F.col("TRIPS") / F.sum("TRIPS").over(w_all), 2))
      .sort("PASSENGER_COUNT"))

timed_show("l", ql)
print("✓ Pregunta (l) completada")
print("=" * 70)

(l) Distribución de passenger_count y efecto en total_amount
----------------------------------------------------------------------------------------
|"PASSENGER_COUNT"  |"TRIPS"    |"AVG_TOTAL_AMOUNT"  |"AVG_DISTANCE"  |"PCT_OF_TOTAL"  |
----------------------------------------------------------------------------------------
|0.0                |5891588    |19.86               |2.77            |0.74            |
|1.0                |579661251  |18.4                |4.03            |73.13           |
|2.0                |111777497  |19.87               |4.25            |14.10           |
|3.0                |30846870   |19.31               |3.81            |3.89            |
|4.0                |15033311   |20.35               |3.86            |1.90            |
|5.0                |30677528   |17.15               |3.05            |3.87            |
|6.0                |18749831   |17.03               |3.01            |2.37            |
|7.0                |3595       |45.54           

## (m) Impacto de tolls_amount y congestion_surcharge por zona

In [52]:
print("=" * 70)
print("(m) Impacto de tolls y congestion_surcharge por zona")
print("=" * 70)

qm = (obt
      .filter(F.col("PU_ZONE").is_not_null())
      .group_by("PU_ZONE")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("TOLLS_AMOUNT"), 2).alias("AVG_TOLLS"),
          F.round(F.sum("TOLLS_AMOUNT"), 0).alias("TOTAL_TOLLS"),
          F.round(F.avg("CONGESTION_SURCHARGE"), 2).alias("AVG_CONGESTION"),
          F.round(F.sum("CONGESTION_SURCHARGE"), 0).alias("TOTAL_CONGESTION"),
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TOTAL")
      )
      .sort(F.col("TOTAL_CONGESTION").desc())
      .limit(20))

timed_show("m", qm, 20)
print("✓ Pregunta (m) completada")
print("=" * 70)

(m) Impacto de tolls y congestion_surcharge por zona
-------------------------------------------------------------------------------------------------------------------------------
|"PU_ZONE"                     |"TRIPS"   |"AVG_TOLLS"  |"TOTAL_TOLLS"  |"AVG_CONGESTION"  |"TOTAL_CONGESTION"  |"AVG_TOTAL"  |
-------------------------------------------------------------------------------------------------------------------------------
|Upper East Side South         |31133959  |0.07         |2242185.0      |2.43              |32634477.0          |14.44        |
|Midtown Center                |29022570  |0.28         |8176280.0      |2.41              |29345623.0          |17.64        |
|Upper East Side North         |28307063  |0.1          |2899363.0      |2.43              |29039247.0          |14.9         |
|Penn Station/Madison Sq West  |25456622  |0.19         |4821549.0      |2.41              |24182335.0          |17.15        |
|Midtown East                  |25297839  |0.26    

## (n) Proporción de viajes cortos vs largos por borough y estacionalidad

In [53]:
print("=" * 70)
print("(n) Viajes cortos vs largos por borough y estacionalidad")
print("=" * 70)

qn_base = (obt
    .filter(F.col("PU_BOROUGH").is_not_null() & F.col("TRIP_DISTANCE").is_not_null())
    .with_column("SEASON",
        F.when(F.col("MONTH").in_(F.lit(12), F.lit(1), F.lit(2)), F.lit("Winter"))
         .when(F.col("MONTH").in_(F.lit(3), F.lit(4), F.lit(5)), F.lit("Spring"))
         .when(F.col("MONTH").in_(F.lit(6), F.lit(7), F.lit(8)), F.lit("Summer"))
         .otherwise(F.lit("Fall")))
    .with_column("DISTANCE_CATEGORY",
        F.when(F.col("TRIP_DISTANCE") <= 2, F.lit("Short (<=2 mi)"))
         .when(F.col("TRIP_DISTANCE") <= 10, F.lit("Medium (2-10 mi)"))
         .otherwise(F.lit("Long (>10 mi)"))))

w_season = Window.partition_by("PU_BOROUGH", "SEASON")

qn = (qn_base
      .group_by("PU_BOROUGH", "SEASON", "DISTANCE_CATEGORY")
      .agg(F.count("*").alias("TRIPS"))
      .with_column("PCT_SHARE",
          F.round(100.0 * F.col("TRIPS") / F.sum("TRIPS").over(w_season), 2))
      .sort("PU_BOROUGH", "SEASON", "DISTANCE_CATEGORY"))

timed_show("n", qn, 100)
print("✓ Pregunta (n) completada")
print("=" * 70)

(n) Viajes cortos vs largos por borough y estacionalidad
----------------------------------------------------------------------------
|"PU_BOROUGH"   |"SEASON"  |"DISTANCE_CATEGORY"  |"TRIPS"    |"PCT_SHARE"  |
----------------------------------------------------------------------------
|Bronx          |Fall      |Long (>10 mi)        |162664     |13.57        |
|Bronx          |Fall      |Medium (2-10 mi)     |571355     |47.66        |
|Bronx          |Fall      |Short (<=2 mi)       |464685     |38.77        |
|Bronx          |Spring    |Long (>10 mi)        |158215     |10.48        |
|Bronx          |Spring    |Medium (2-10 mi)     |716685     |47.47        |
|Bronx          |Spring    |Short (<=2 mi)       |634998     |42.06        |
|Bronx          |Summer    |Long (>10 mi)        |152870     |12.00        |
|Bronx          |Summer    |Medium (2-10 mi)     |602518     |47.29        |
|Bronx          |Summer    |Short (<=2 mi)       |518617     |40.71        |
|Bronx          |Wi

## (o) Diferencias por vendor en avg_speed_mph y trip_duration_min

In [54]:
print("=" * 70)
print("(o) Diferencias por vendor en velocidad y duración")
print("=" * 70)

qo = (obt
      .filter(F.col("VENDOR_NAME").is_not_null())
      .group_by("VENDOR_NAME")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("AVG_SPEED_MPH"), 2).alias("AVG_SPEED"),
          F.round(F.avg("TRIP_DURATION_MIN"), 2).alias("AVG_DURATION"),
          F.round(F.avg("TRIP_DISTANCE"), 2).alias("AVG_DISTANCE"),
          F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TOTAL")
      )
      .sort(F.col("TRIPS").desc()))

timed_show("o", qo)
print("✓ Pregunta (o) completada")
print("=" * 70)

(o) Diferencias por vendor en velocidad y duración
----------------------------------------------------------------------------------------------------------
|"VENDOR_NAME"                 |"TRIPS"    |"AVG_SPEED"  |"AVG_DURATION"  |"AVG_DISTANCE"  |"AVG_TOTAL"  |
----------------------------------------------------------------------------------------------------------
|VeriFone Inc.                 |513184982  |17.1         |21.56           |4.47            |19.41        |
|Creative Mobile Technologies  |300797203  |20.75        |13.61           |5.34            |17.78        |
|Unknown                       |1571665    |10.68        |12.63           |3.68            |23.88        |
----------------------------------------------------------------------------------------------------------

⏱  Tiempo de ejecución (o): 0.13 s
✓ Pregunta (o) completada


## (p) Relación método de pago ↔ tip_amount por hora

In [55]:
print("=" * 70)
print("(p) Relación método de pago y tip_amount por hora")
print("=" * 70)

qp = (obt
      .filter(F.col("PAYMENT_TYPE_DESC").is_not_null())
      .group_by("PAYMENT_TYPE_DESC", "PICKUP_HOUR")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg("TIP_AMOUNT"), 2).alias("AVG_TIP"),
          F.round(F.avg("TIP_PCT"), 2).alias("AVG_TIP_PCT")
      )
      .sort("PAYMENT_TYPE_DESC", "PICKUP_HOUR"))

timed_show("p", qp, 100)
print("✓ Pregunta (p) completada")
print("=" * 70)

(p) Relación método de pago y tip_amount por hora
------------------------------------------------------------------------------
|"PAYMENT_TYPE_DESC"  |"PICKUP_HOUR"  |"TRIPS"   |"AVG_TIP"  |"AVG_TIP_PCT"  |
------------------------------------------------------------------------------
|Cash                 |0              |7595724   |0.0        |0.0            |
|Cash                 |1              |5572082   |0.0        |0.0            |
|Cash                 |2              |4100166   |0.0        |0.0            |
|Cash                 |3              |3193479   |0.0        |0.0            |
|Cash                 |4              |2937215   |0.0        |0.0            |
|Cash                 |5              |2656506   |0.0        |0.0            |
|Cash                 |6              |4880376   |0.0        |0.0            |
|Cash                 |7              |7222808   |0.0        |0.0            |
|Cash                 |8              |8708240   |0.0        |0.0            |
|C

## (q) Zonas con percentil 99 de duración/distancia fuera de rango (posible congestión/eventos)

In [56]:
print("=" * 70)
print("(q) Zonas con p99 de duración/distancia fuera de rango")
print("=" * 70)

# Calcular umbrales p99 con SQL nativo via session.sql
thresholds = session.sql("""
    SELECT
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY TRIP_DURATION_MIN) AS P99_DURATION,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY TRIP_DISTANCE) AS P99_DISTANCE
    FROM OBT_TRIPS
    WHERE TRIP_DURATION_MIN >= 0 AND TRIP_DISTANCE >= 0
""").collect()[0]

p99_dur = thresholds["P99_DURATION"]
p99_dist = thresholds["P99_DISTANCE"]
print(f"  Umbrales p99: duración={p99_dur} min, distancia={p99_dist} mi")

qq = (obt
      .filter(
          F.col("PU_ZONE").is_not_null() &
          ((F.col("TRIP_DURATION_MIN") > F.lit(p99_dur)) | (F.col("TRIP_DISTANCE") > F.lit(p99_dist)))
      )
      .group_by("PU_ZONE", "PU_BOROUGH")
      .agg(
          F.count("*").alias("EXTREME_TRIPS"),
          F.round(F.avg("TRIP_DURATION_MIN"), 2).alias("AVG_DURATION"),
          F.round(F.avg("TRIP_DISTANCE"), 2).alias("AVG_DISTANCE")
      )
      .sort(F.col("EXTREME_TRIPS").desc())
      .limit(20))

timed_show("q", qq, 20)
print("✓ Pregunta (q) completada")
print("=" * 70)

(q) Zonas con p99 de duración/distancia fuera de rango
  Umbrales p99: duración=64.000 min, distancia=19.13 mi
---------------------------------------------------------------------------------------------------
|"PU_ZONE"                     |"PU_BOROUGH"  |"EXTREME_TRIPS"  |"AVG_DURATION"  |"AVG_DISTANCE"  |
---------------------------------------------------------------------------------------------------
|JFK Airport                   |Queens        |6752887          |87.22           |28.53           |
|LaGuardia Airport             |Queens        |739266           |484.17          |49.48           |
|Times Sq/Theatre District     |Manhattan     |321409           |354.61          |173.78          |
|Midtown Center                |Manhattan     |230550           |377.84          |117.98          |
|Midtown North                 |Manhattan     |192573           |292.60          |31.72           |
|Clinton East                  |Manhattan     |175316           |480.08          |198.36 

## (r) Yield por milla (total_amount / trip_distance) por borough y hora

In [57]:
print("=" * 70)
print("(r) Yield por milla por borough y hora")
print("=" * 70)

qr = (obt
      .filter(
          F.col("PU_BOROUGH").is_not_null() &
          (F.col("TRIP_DISTANCE") > 0) &
          (F.col("TOTAL_AMOUNT") > 0)
      )
      .group_by("PU_BOROUGH", "PICKUP_HOUR")
      .agg(
          F.count("*").alias("TRIPS"),
          F.round(F.avg(F.col("TOTAL_AMOUNT") / F.col("TRIP_DISTANCE")), 2).alias("YIELD_PER_MILE")
      )
      .sort("PU_BOROUGH", "PICKUP_HOUR"))

timed_show("r", qr, 100)
print("✓ Pregunta (r) completada")
print("=" * 70)

(r) Yield por milla por borough y hora
--------------------------------------------------------------
|"PU_BOROUGH"  |"PICKUP_HOUR"  |"TRIPS"   |"YIELD_PER_MILE"  |
--------------------------------------------------------------
|Bronx         |0              |131442    |14.31             |
|Bronx         |1              |96238     |16.92             |
|Bronx         |2              |68318     |18.21             |
|Bronx         |3              |58648     |16.17             |
|Bronx         |4              |74687     |14.09             |
|Bronx         |5              |93823     |12.6              |
|Bronx         |6              |162386    |11.62             |
|Bronx         |7              |264651    |11.23             |
|Bronx         |8              |323583    |11.93             |
|Bronx         |9              |298459    |12.04             |
|Bronx         |10             |268020    |11.87             |
|Bronx         |11             |256018    |11.73             |
|Bronx         |

## (s) Cambios YoY en volumen y ticket promedio por service_type

In [58]:
print("=" * 70)
print("(s) Cambios YoY en volumen y ticket promedio")
print("=" * 70)

w_yoy = Window.partition_by("SERVICE_TYPE").order_by("YEAR")

yearly = (obt
    .group_by("SERVICE_TYPE", "YEAR")
    .agg(
        F.count("*").alias("TRIPS"),
        F.round(F.avg("TOTAL_AMOUNT"), 2).alias("AVG_TICKET")
    ))

qs = (yearly
      .with_column("PREV_TRIPS", F.lag("TRIPS", 1).over(w_yoy))
      .with_column("PREV_TICKET", F.lag("AVG_TICKET", 1).over(w_yoy))
      .with_column("YOY_VOLUME_PCT",
          F.round(100.0 * (F.col("TRIPS") - F.col("PREV_TRIPS")) / F.col("PREV_TRIPS"), 2))
      .with_column("YOY_TICKET_CHANGE",
          F.round(F.col("AVG_TICKET") - F.col("PREV_TICKET"), 2))
      .select("SERVICE_TYPE", "YEAR", "TRIPS", "AVG_TICKET", "YOY_VOLUME_PCT", "YOY_TICKET_CHANGE")
      .sort("SERVICE_TYPE", "YEAR"))

timed_show("s", qs, 50)
print("✓ Pregunta (s) completada")
print("=" * 70)

(s) Cambios YoY en volumen y ticket promedio
-----------------------------------------------------------------------------------------------
|"SERVICE_TYPE"  |"YEAR"  |"TRIPS"    |"AVG_TICKET"  |"YOY_VOLUME_PCT"  |"YOY_TICKET_CHANGE"  |
-----------------------------------------------------------------------------------------------
|green           |2015    |19233765   |14.84         |NULL              |NULL                 |
|green           |2016    |16385541   |14.64         |-14.81            |-0.2                 |
|green           |2017    |11737059   |14.24         |-28.37            |-0.4                 |
|green           |2018    |8899718    |16.09         |-24.17            |1.85                 |
|green           |2019    |6300985    |18.33         |-29.20            |2.24                 |
|green           |2020    |1734176    |20.16         |-72.48            |1.83                 |
|green           |2021    |1068755    |23.93         |-38.37            |3.77              

## (t) Días con alta congestion_surcharge: efecto en total_amount vs días "normales"

In [59]:
print("=" * 70)
print("(t) Días alta congestion vs normales: efecto en total_amount")
print("=" * 70)

daily_avg = (obt
    .filter(F.col("PICKUP_DATE").is_not_null() & F.col("CONGESTION_SURCHARGE").is_not_null())
    .group_by("PICKUP_DATE")
    .agg(
        F.avg("CONGESTION_SURCHARGE").alias("AVG_DAILY_CONGESTION"),
        F.avg("TOTAL_AMOUNT").alias("AVG_DAILY_TOTAL"),
        F.count("*").alias("TRIPS")
    ))

# Calcular p90 de congestion diaria
p90_cong = session.sql("""
    SELECT PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY AVG_DAILY_CONGESTION) AS P90
    FROM (
        SELECT PICKUP_DATE, AVG(CONGESTION_SURCHARGE) AS AVG_DAILY_CONGESTION
        FROM OBT_TRIPS
        WHERE PICKUP_DATE IS NOT NULL AND CONGESTION_SURCHARGE IS NOT NULL
        GROUP BY PICKUP_DATE
    )
""").collect()[0]["P90"]

print(f"  Umbral p90 congestion diaria: {p90_cong}")

qt = (daily_avg
      .with_column("DAY_TYPE",
          F.when(F.col("AVG_DAILY_CONGESTION") >= F.lit(p90_cong), F.lit("High Congestion Day"))
           .otherwise(F.lit("Normal Day")))
      .group_by("DAY_TYPE")
      .agg(
          F.count("*").alias("NUM_DAYS"),
          F.round(F.avg("AVG_DAILY_TOTAL"), 2).alias("AVG_TOTAL_AMOUNT"),
          F.round(F.avg("AVG_DAILY_CONGESTION"), 2).alias("AVG_CONGESTION_SURCHARGE"),
          F.round(F.avg("TRIPS"), 0).alias("AVG_DAILY_TRIPS")
      )
      .sort("DAY_TYPE"))

timed_show("t", qt)
print("✓ Pregunta (t) completada")
print("=" * 70)

(t) Días alta congestion vs normales: efecto en total_amount
  Umbral p90 congestion diaria: 2.291664687500721
----------------------------------------------------------------------------------------------------------
|"DAY_TYPE"           |"NUM_DAYS"  |"AVG_TOTAL_AMOUNT"  |"AVG_CONGESTION_SURCHARGE"  |"AVG_DAILY_TRIPS"  |
----------------------------------------------------------------------------------------------------------
|High Congestion Day  |261         |23.12               |2.32                        |107107             |
|Normal Day           |2341        |22.94               |2.15                        |111594             |
----------------------------------------------------------------------------------------------------------

⏱  Tiempo de ejecución (t): 0.16 s
✓ Pregunta (t) completada


---
## Resumen de Performance (Sección 12)

Tiempos de ejecución de cada pregunta **sin índices ni clustering** en `analytics.OBT_TRIPS`.
Documenta el rendimiento base según la sección 12 del enunciado.

In [60]:
print("=" * 70)
print("RESUMEN DE PERFORMANCE — SIN ÍNDICES NI CLUSTERING")
print("=" * 70)

total_time = sum(timings.values())

print(f"\n{'Pregunta':<12} {'Tiempo (s)':>12}")
print("-" * 26)
for k in sorted(timings.keys()):
    print(f"  ({k}){' ' * (8 - len(k))}{timings[k]:>10.2f} s")
print("-" * 26)
print(f"  TOTAL{' ' * 5}{total_time:>10.2f} s")
print(f"  PROMEDIO{' ' * 2}{total_time / len(timings):>10.2f} s")
print()
print("Nota: OBT_TRIPS NO tiene índices ni clustering keys.")
print("      Estos tiempos reflejan el rendimiento base de Snowflake")
print("      con micro-partitioning automático únicamente.")
print("=" * 70)

session.close()
print("✓ Sesión Snowpark cerrada. NB05 FINALIZADO.")

RESUMEN DE PERFORMANCE — SIN ÍNDICES NI CLUSTERING

Pregunta       Tiempo (s)
--------------------------
  (a)             0.15 s
  (b)             0.16 s
  (c)             0.16 s
  (d)             0.19 s
  (e)             0.16 s
  (f)             9.41 s
  (g)             0.16 s
  (h)             0.25 s
  (i)             0.15 s
  (j)             0.27 s
  (k)             0.15 s
  (l)             0.25 s
  (m)             0.15 s
  (n)             0.31 s
  (o)             0.13 s
  (p)             0.17 s
  (q)             0.15 s
  (r)             0.14 s
  (s)             0.14 s
  (t)             0.16 s
--------------------------
  TOTAL          12.81 s
  PROMEDIO        0.64 s

Nota: OBT_TRIPS NO tiene índices ni clustering keys.
      Estos tiempos reflejan el rendimiento base de Snowflake
      con micro-partitioning automático únicamente.
✓ Sesión Snowpark cerrada. NB05 FINALIZADO.


---
## Resumen

Se respondieron las **20 preguntas de negocio** usando **Snowpark DataFrames** contra `ANALYTICS.OBT_TRIPS`.
Todas las operaciones se ejecutan dentro del motor de Snowflake.

### Notas de Performance (Sección 12)
- **Sin índices** creados en la tabla OBT.
- **Sin clustering keys** en la tabla OBT.
- Snowflake utiliza únicamente su **micro-partitioning automático**.
- Los tiempos registrados representan el **rendimiento base** sin optimizaciones manuales.
- Si se añadieran clustering keys (e.g. `service_type, year, month`), se esperaría una mejora en consultas que filtran por esas columnas.